In [2]:
import tensorflow as tf
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)
from tensorflow.keras import Model

# -------------------------------
# Dataset (English → Telugu)
# Using transliterated Telugu for ASCII-safe tokenization.
# You can replace with actual Telugu Unicode strings too.
# -------------------------------

data = [
    ("i am a student",        "nenu vidyarthi ని"),
    ("how are you",           "meeru ela unnaru"),
    ("i love machine learning","nenu machine learning ishtapadatanu"),
    ("good morning",          "subhodayam"),
    ("thank you",             "dhanyavadalu"),
    ("see you later",         "tarvata kalustanu"),
    ("what is your name",     "mee peru emi"),
    ("where are you going",   "meeru ekkadi veltunnaru"),
    ("i like coffee",         "nenu coffee ishtapadatanu"),
    ("welcome",               "swagatum"),
]

english_sentences = [x[0] for x in data]
telugu_sentences  = ["start " + x[1] + " end" for x in data]

# -------------------------------
# Tokenization
# -------------------------------

vocab_size      = 2000   # larger to handle Telugu tokens
sequence_length = 12     # slightly longer for Telugu phrases

source_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

target_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

source_vectorization.adapt(english_sentences)
target_vectorization.adapt(telugu_sentences)

encoder_inputs_data = source_vectorization(english_sentences)
target_tokens       = target_vectorization(telugu_sentences)

decoder_inputs_data = target_tokens[:, :-1]
decoder_targets     = target_tokens[:, 1:]

# -------------------------------
# Positional Embedding
# -------------------------------

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding    = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embedding = Embedding(input_dim=sequence_length, output_dim=embed_dim)

    def call(self, inputs):
        length             = tf.shape(inputs)[-1]
        positions          = tf.range(start=0, limit=length, delta=1)
        token_embeddings   = self.token_embedding(inputs)
        position_embeddings = self.position_embedding(positions)
        return token_embeddings + position_embeddings

# -------------------------------
# Encoder
# -------------------------------

class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()
        self.attention   = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj  = tf.keras.Sequential([Dense(dense_dim, activation="relu"), Dense(embed_dim)])
        self.layernorm1  = LayerNormalization()
        self.layernorm2  = LayerNormalization()

    def call(self, inputs):
        attention_output = self.attention(query=inputs, value=inputs, key=inputs)
        proj_input       = self.layernorm1(inputs + attention_output)
        proj_output      = self.dense_proj(proj_input)
        return self.layernorm2(proj_input + proj_output)

# -------------------------------
# Decoder
# -------------------------------

class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()
        self.self_attention  = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.cross_attention = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn             = tf.keras.Sequential([Dense(dense_dim, activation="relu"), Dense(embed_dim)])
        self.layernorm1      = LayerNormalization()
        self.layernorm2      = LayerNormalization()
        self.layernorm3      = LayerNormalization()

    def call(self, inputs, encoder_outputs):
        attention_output  = self.self_attention(query=inputs, value=inputs, key=inputs, use_causal_mask=True)
        out1              = self.layernorm1(inputs + attention_output)
        attention_output2 = self.cross_attention(query=out1, value=encoder_outputs, key=encoder_outputs)
        out2              = self.layernorm2(out1 + attention_output2)
        ffn_output        = self.ffn(out2)
        return self.layernorm3(out2 + ffn_output)

# -------------------------------
# Build Transformer
# -------------------------------

embed_dim  = 128
dense_dim  = 256
num_heads  = 4

encoder_input = tf.keras.Input(shape=(None,), dtype="int64")
decoder_input = tf.keras.Input(shape=(None,), dtype="int64")

encoder_embeddings = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_input)
encoder_output     = TransformerEncoder(embed_dim, dense_dim, num_heads)(encoder_embeddings)

decoder_embeddings = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_input)
decoder_output     = TransformerDecoder(embed_dim, dense_dim, num_heads)(decoder_embeddings, encoder_output)

final_output = Dense(vocab_size, activation="softmax")(decoder_output)

transformer = Model([encoder_input, decoder_input], final_output)

# -------------------------------
# Compile
# -------------------------------

transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

transformer.summary()

# -------------------------------
# Train
# -------------------------------

history = transformer.fit(
    [encoder_inputs_data, decoder_inputs_data],
    decoder_targets,
    epochs=100,          # more epochs — small dataset needs longer training
    batch_size=2,
    verbose=1
)

# -------------------------------
# Final Metrics
# -------------------------------

final_acc  = history.history['accuracy'][-1]
final_loss = history.history['loss'][-1]
print(f"\nFinal Accuracy : {round(final_acc * 100, 2)} %")
print(f"Final Loss     : {round(final_loss, 4)}")

# -------------------------------
# Inference Helper
# -------------------------------

target_vocab     = target_vectorization.get_vocabulary()
target_index_lookup = dict(zip(range(len(target_vocab)), target_vocab))

def translate(input_sentence: str, max_length: int = sequence_length) -> str:
    """Greedy-decode a Telugu translation for the given English sentence."""
    tokenized_src = source_vectorization([input_sentence])           # (1, seq_len)
    decoded_tokens = ["start"]

    for _ in range(max_length):
        tokenized_tgt = target_vectorization([" ".join(decoded_tokens)])  # (1, seq_len)
        predictions   = transformer([tokenized_src, tokenized_tgt], training=False)  # (1, seq, vocab)

        # Take the last generated position
        next_token_idx = tf.argmax(predictions[0, len(decoded_tokens) - 1, :]).numpy()
        next_token     = target_index_lookup.get(next_token_idx, "")

        if next_token == "end" or next_token == "":
            break
        decoded_tokens.append(next_token)

    return " ".join(decoded_tokens[1:])   # strip "start"


# -------------------------------
# Test Translations
# -------------------------------

test_sentences = [
    "i am a student",
    "good morning",
    "thank you",
    "how are you",
    "welcome",
]

print("\n--- Translations ---")
for sentence in test_sentences:
    print(f"  EN: {sentence}")
    print(f"  TE: {translate(sentence)}\n")

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │    257,536 │ input_layer[0][0] │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │    257,536 │ input_layer_1[0]… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, None, 128) │    330,240 │ positional_embed… │
│ (TransformerEncode… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_decoder │ (None, None, 128) │    594,304 │ positional_embed… │
│ (TransformerDecode… │                   │            │ transformer_enco… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, None,      │    258,000 │ transformer_deco… │
│                     │ 2000)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,697,616 (6.48 MB)

 Trainable params: 1,697,616 (6.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.5455 - loss: 5.8455   
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6909 - loss: 4.0663
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6909 - loss: 3.0980
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6909 - loss: 2.3091
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6909 - loss: 1.7909
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7000 - loss: 1.4361
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7273 - loss: 1.2016
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7636 - loss: 1.0283
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7818 - loss: 0.8997
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7818 - loss: 0.7894
Epoch 11/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8273 - loss: 0.7000
Epoch 12/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8364 -